# Sanskrit Probing Dataset Builder — V1
**Goal:** Extract contextual activations from the trained Sanskrit char-LM, aligned to DCS morphological (case) labels.  
**Output:** Activation vectors for every case-bearing nominal in the DCS eval files, paired with case labels.  
**The cardinal rule:** alignment. Each activation must come from the exact character positions of the word whose label it carries.

**Pipeline:**
1. Parse DCS CoNLL-U → SLP1 sentences with exact char-span offsets per token  
2. Count per-class tokens → **stop and report before spending GPU time**  
3. Load trained model  
4. Extract contextual activations (`final_char` + `mean_pool`, all layers)  
5. Save dataset + manifest  

> Run cells top to bottom. Cell 4 is a checkpoint — review counts before proceeding.

---
## Cell 0 · Config
All paths and knobs in one place.

In [ ]:
# ── PATHS ──────────────────────────────────────────────────────────────────────
# Directory containing subdirs of .conllu files (one subdir = one DCS work).
# Upload sanskrit_corpus/data/raw/dcs/sanskrit/dcs/data/conllu/files/ to Drive.
DCS_CONLLU_PATH = '/content/drive/MyDrive/sanskrit/dcs/conllu'

MODEL_CKPT_PATH = '/content/drive/MyDrive/sanskrit/v1_run/best_checkpoint.pt'
VOCAB_PATH      = '/content/drive/MyDrive/sanskrit/v1_run/vocab.json'
OUTPUT_DIR      = '/content/drive/MyDrive/sanskrit/probing_v1'

# ── EXTRACTION ─────────────────────────────────────────────────────────────────
# Which layers to extract (0-indexed). None → all layers (set after model load).
LAYERS            = None
POSITION_VARIANTS = ['final_char', 'mean_pool']

# Set to an int (e.g. 500) for a quick smoke-test; None = process everything.
MAX_SENTENCES = None

# ── QUALITY ────────────────────────────────────────────────────────────────────
# Case classes below this token count are flagged as thin / unreliable.
THIN_CLASS_THRESHOLD = 200

print('Config loaded.')
print(f'  DCS path   : {DCS_CONLLU_PATH}')
print(f'  Checkpoint : {MODEL_CKPT_PATH}')
print(f'  Output     : {OUTPUT_DIR}')

---
## Cell 1 · Mount Drive · Install · Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, importlib
if importlib.util.find_spec('indic_transliteration') is None:
    subprocess.run(['pip', 'install', '-q', 'indic-transliteration'], check=True)

import os, re, json, csv, time, traceback
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'Output : {OUTPUT_DIR}')

---
## Cell 2 · Helper Functions
CoNLL-U parser, transliteration, span computation, vocab-aligned encoding.

**Alignment design:** We build the SLP1 sentence by converting each token's IAST FORM to SLP1
individually and joining with spaces. Spans are computed by byte-offset accumulation over
these SLP1 forms — no searching within the sentence string. This is exact by construction.
IAST digraphs (bh→B, etc.) shrink on conversion, so we must convert before accumulating.

In [ ]:
# ── Transliteration ────────────────────────────────────────────────────────────
# DCS forms are IAST (diacritics: ā ī ū ṃ ḥ …). Devanagari detection is a fallback.
_DEVA_RE = re.compile(r'[\u0900-\u097F]')

def to_slp1(text: str) -> str:
    """Convert IAST (or Devanagari) text to SLP1."""
    if not text:
        return ''
    scheme = sanscript.DEVANAGARI if _DEVA_RE.search(text) else sanscript.IAST
    return transliterate(text, scheme, sanscript.SLP1)


# ── CoNLL-U field parsers ──────────────────────────────────────────────────────
def _parse_kv(s: str) -> dict:
    """'Case=Nom|Number=Sing' → {'Case': 'Nom', 'Number': 'Sing'}. '_' → {}."""
    if not s or s.strip() == '_':
        return {}
    out = {}
    for pair in s.split('|'):
        if '=' in pair:
            k, v = pair.split('=', 1)
            out[k.strip()] = v.strip()
    return out


# ── CoNLL-U file parser ────────────────────────────────────────────────────────
def parse_conllu_file(path) -> list:
    """
    Parse one DCS CoNLL-U file.
    Returns list of sentence dicts: {sent_id, text, source_file, tokens}
    Each token: {id, id_int, form, lemma, upos, feats, misc,
                 is_mwt, is_mwt_sub, mwt_start, mwt_end}
    """
    sentences = []
    cur = None
    active_mwt_end = None  # last sub-token id of the current MWT group
    src = str(path)

    with open(path, encoding='utf-8', errors='replace') as f:
        lines = f.read().splitlines()

    for line in lines:
        stripped = line.strip()

        if not stripped:  # blank line = sentence boundary
            if cur is not None:
                sentences.append(cur)
            cur = None
            active_mwt_end = None
            continue

        if stripped.startswith('##'):
            continue

        if stripped.startswith('#'):
            if cur is None:
                cur = {'sent_id': '', 'text': '', 'source_file': src, 'tokens': []}
            if stripped.startswith('# sent_id = '):
                cur['sent_id'] = stripped[len('# sent_id = '):]
            elif stripped.startswith('# text = '):
                cur['text'] = stripped[len('# text = '):]
            continue

        # Token / MWT line
        parts = stripped.split('\t')
        if len(parts) < 10:
            continue
        if cur is None:
            cur = {'sent_id': '', 'text': '', 'source_file': src, 'tokens': []}

        tid   = parts[0]
        form  = parts[1]
        lemma = parts[2]
        upos  = parts[3]
        feats = _parse_kv(parts[5])
        misc  = _parse_kv(parts[9])

        if '-' in tid:  # MWT line
            mwt_s, mwt_e = (int(x) for x in tid.split('-', 1))
            active_mwt_end = mwt_e
            cur['tokens'].append({
                'id': tid, 'id_int': None,
                'form': form, 'lemma': lemma, 'upos': upos,
                'feats': feats, 'misc': misc,
                'is_mwt': True, 'is_mwt_sub': False,
                'mwt_start': mwt_s, 'mwt_end': mwt_e,
            })
        else:
            try:
                id_int = int(tid)
            except ValueError:
                continue
            is_sub = (active_mwt_end is not None and id_int <= active_mwt_end)
            if active_mwt_end is not None and id_int >= active_mwt_end:
                active_mwt_end = None
            cur['tokens'].append({
                'id': tid, 'id_int': id_int,
                'form': form, 'lemma': lemma, 'upos': upos,
                'feats': feats, 'misc': misc,
                'is_mwt': False, 'is_mwt_sub': is_sub,
                'mwt_start': None, 'mwt_end': None,
            })

    if cur is not None:
        sentences.append(cur)

    return sentences


# ── Sentence builder with span tracking ───────────────────────────────────────
# Each case in DCS: Nom Acc Ins Dat Abl Gen Loc Voc (+ Cpd = compound element, skip)
_VALID_CASES = {'Nom', 'Acc', 'Ins', 'Dat', 'Abl', 'Gen', 'Loc', 'Voc'}

def build_slp1_sentence(sentence: dict):
    """
    Build the SLP1 sentence string and compute char-span offsets.

    Strategy:
      - MWT tokens contribute their (fused) surface FORM to the string.
        Their sub-tokens are context-only; they have NO standalone char span
        (sandhi fusion makes sub-token character boundaries ambiguous).
        Sub-tokens with Case features are logged as skipped.
      - Regular (non-sub) tokens each contribute their FORM to the string.
        Their span is computed by accumulation — exact by construction.

    Returns:
      slp1_text  : str — space-joined SLP1 forms for all surface tokens
      targets    : list of target-token dicts with char_start / char_end
      skipped    : list of (token_id, form, reason) for logged skips
    """
    surface_parts = []  # (tok_or_None, form_slp1) for the string
    skipped = []

    for tok in sentence['tokens']:
        if tok['is_mwt']:
            # MWT: add its fused surface form; sub-tokens that follow won't appear
            try:
                form_slp1 = to_slp1(tok['form'])
            except Exception as e:
                skipped.append((tok['id'], tok['form'], f'transliteration_error: {e}'))
                form_slp1 = None
            surface_parts.append((None, form_slp1))  # None = not a probing target

        elif tok['is_mwt_sub']:
            # Sub-token: context only, no surface span. Log if it has Case.
            case = tok['feats'].get('Case', '')
            if case in _VALID_CASES:
                skipped.append((tok['id'], tok['form'],
                                f'mwt_sub (case={case}): inside sandhi-fused MWT, '  
                                f'char boundary ambiguous'))
        else:
            # Regular token: contributes its own surface form
            try:
                form_slp1 = to_slp1(tok['form'])
            except Exception as e:
                skipped.append((tok['id'], tok['form'], f'transliteration_error: {e}'))
                form_slp1 = None
            surface_parts.append((tok, form_slp1))

    # Accumulate spans and build sentence string
    slp1_forms = []
    targets = []
    offset = 0

    for tok, form_slp1 in surface_parts:
        if form_slp1 is None:
            continue  # transliteration failed for this surface token — skip
        start = offset
        end   = offset + len(form_slp1)  # exclusive
        slp1_forms.append(form_slp1)
        offset = end + 1  # +1 for the space separator

        if tok is None:
            continue  # MWT fused form: contributes to string but not a probe target

        case = tok['feats'].get('Case', '')
        if case in _VALID_CASES:
            targets.append({
                'sent_id'    : sentence['sent_id'],
                'source_file': sentence['source_file'],
                'token_id'   : tok['id'],
                'token_idx'  : tok['id_int'],
                'form_iast'  : tok['form'],
                'form_slp1'  : form_slp1,
                'lemma'      : tok['lemma'],
                'upos'       : tok['upos'],
                'case_label' : case,
                'full_feats' : '|'.join(f'{k}={v}' for k, v in tok['feats'].items()),
                'char_start' : start,
                'char_end'   : end,
            })

    slp1_text = ' '.join(slp1_forms)
    return slp1_text, targets, skipped


# ── Vocab-aligned encoding ─────────────────────────────────────────────────────
def encode_aligned(slp1_text: str, stoi: dict):
    """
    Encode SLP1 string → list of token IDs, plus a char→position mapping.
    Characters absent from the vocab are dropped; their mapping entry is -1.
    """
    ids = []
    char_to_pos = []
    for ch in slp1_text:
        if ch in stoi:
            char_to_pos.append(len(ids))
            ids.append(stoi[ch])
        else:
            char_to_pos.append(-1)
    return ids, char_to_pos


def span_to_positions(char_start, char_end, char_to_pos):
    """
    Map a char span [start, end) in SLP1 string to encoded-sequence positions.
    Returns (tok_start, tok_end_inclusive) or None if the span has no vocab chars.
    """
    first = None
    for i in range(char_start, min(char_end, len(char_to_pos))):
        if char_to_pos[i] != -1:
            first = char_to_pos[i]
            break
    if first is None:
        return None

    last = None
    for i in range(min(char_end, len(char_to_pos)) - 1, char_start - 1, -1):
        if char_to_pos[i] != -1:
            last = char_to_pos[i]
            break
    if last is None:
        return None

    return first, last  # inclusive positions in the encoded sequence


# ── Sanity-check display ───────────────────────────────────────────────────────
def highlight_tokens(slp1_text, targets, max_show=3):
    """Print sentence with target token spans highlighted by >>>...<<<."""
    sorted_t = sorted(targets[:max_show], key=lambda t: t['char_start'])
    result = slp1_text
    # Insert markers from right to left so earlier spans' offsets stay valid
    for t in sorted(sorted_t, key=lambda t: t['char_start'], reverse=True):
        s, e = t['char_start'], t['char_end']
        result = result[:s] + '>>>' + result[s:e] + f'<<<[{t["case_label"]}]' + result[e:]
    return result

print('Helper functions defined.')

---
## Cell 3 · Parse DCS → Build SLP1 Sentences with Spans
Reads all CoNLL-U files, builds the SLP1 sentence strings, and computes char-span
offsets for every case-bearing nominal. Displays a few sentences with highlighted spans
for human verification before any activations are extracted.

In [ ]:
conllu_root = Path(DCS_CONLLU_PATH)
conllu_files = sorted(conllu_root.rglob('*.conllu'))
print(f'Found {len(conllu_files):,} CoNLL-U files under {conllu_root}')
if not conllu_files:
    raise FileNotFoundError(
        f'No .conllu files found at {DCS_CONLLU_PATH}.\n'
        f'Upload the DCS conllu directory to Drive at that path.'
    )

# ── Parse ──────────────────────────────────────────────────────────────────────
all_parsed_sentences = []   # list of (slp1_text, targets, sent_meta)
skip_log = Counter()        # reason → count
parse_errors = []

n_sentences_total = 0
n_sentences_skipped_too_long = 0
n_targets_total = 0

# We'll fill BLOCK_SIZE from the checkpoint after model load; use 512 for now.
# Sentences longer than this will be skipped (very rare for Sanskrit sentences).
_BLOCK_SIZE_GUARD = 512

t0 = time.time()
for file_idx, path in enumerate(conllu_files):
    try:
        raw_sents = parse_conllu_file(path)
    except Exception as e:
        parse_errors.append((str(path), str(e)))
        continue

    for sent in raw_sents:
        n_sentences_total += 1
        if MAX_SENTENCES is not None and n_sentences_total > MAX_SENTENCES:
            break

        slp1_text, targets, skipped = build_slp1_sentence(sent)

        for _, _, reason in skipped:
            # Normalise reason to a short category for the counter
            if reason.startswith('mwt_sub'):
                skip_log['mwt_sub (case inside MWT)'] += 1
            elif reason.startswith('transliteration'):
                skip_log['transliteration_error'] += 1
            else:
                skip_log[reason[:60]] += 1

        if not slp1_text:
            continue

        # Guard against sentences exceeding context window
        if len(slp1_text) > _BLOCK_SIZE_GUARD:
            n_sentences_skipped_too_long += 1
            skip_log['sentence_exceeds_block_size'] += len(targets)
            continue

        if targets:
            n_targets_total += len(targets)

        all_parsed_sentences.append({
            'slp1_text': slp1_text,
            'targets'  : targets,
            'sent_id'  : sent['sent_id'],
            'iast_text': sent['text'],
        })

    if MAX_SENTENCES is not None and n_sentences_total > MAX_SENTENCES:
        print(f'MAX_SENTENCES={MAX_SENTENCES} reached.')
        break

print(f'Parsed {n_sentences_total:,} sentences from {len(conllu_files):,} files '
      f'in {time.time()-t0:.1f}s')
print(f'Kept   : {len(all_parsed_sentences):,} sentences (have SLP1 text)')
print(f'Targets: {n_targets_total:,} case-bearing nominal tokens')
print(f'Skipped: {sum(skip_log.values()):,} tokens')
for reason, cnt in skip_log.most_common():
    print(f'  {cnt:6,}  {reason}')
if parse_errors:
    print(f'File errors: {len(parse_errors)} (first 5: {parse_errors[:5]})')

# ── Sanity check: display spans for a few sentences ───────────────────────────
print('\n─── SPAN SANITY CHECK ──────────────────────────────────────────────────')
print('Each target word should be bracketed by >>>...<<<[Case].  '
      'If the wrong word is highlighted, alignment is broken.')
print()

shown = 0
for entry in all_parsed_sentences:
    if not entry['targets']:
        continue
    highlighted = highlight_tokens(entry['slp1_text'], entry['targets'], max_show=3)
    print(f"sent_id={entry['sent_id']}")
    print(f"  IAST : {entry['iast_text']}")
    print(f"  SLP1 : {highlighted}")
    for t in entry['targets'][:3]:
        print(f"  token: '{t['form_slp1']}' (IAST: {t['form_iast']}) "
              f"span=[{t['char_start']},{t['char_end']}) case={t['case_label']}")
        # Verify span: the substring at [start, end) should equal form_slp1
        extracted = entry['slp1_text'][t['char_start']:t['char_end']]
        ok = extracted == t['form_slp1']
        print(f"  check: slp1_text[{t['char_start']}:{t['char_end']}] = '{extracted}' "
              f"{'✓' if ok else '✗ MISMATCH — BUG'}")
    print()
    shown += 1
    if shown >= 5:
        break

---
## Cell 4 · Per-Class Count · CHECKPOINT
**Read the table below before proceeding to extraction.**  
Classes below `THIN_CLASS_THRESHOLD` are flagged. If you see thin classes that matter
for your probing hypotheses, decide now whether to merge or drop them rather than
extracting and discovering the issue later.

In [ ]:
from collections import Counter

case_counter = Counter()
for entry in all_parsed_sentences:
    for t in entry['targets']:
        case_counter[t['case_label']] += 1

# Build report table
rows = []
total = sum(case_counter.values())
for case in ['Nom', 'Acc', 'Ins', 'Dat', 'Abl', 'Gen', 'Loc', 'Voc']:
    cnt  = case_counter.get(case, 0)
    pct  = 100.0 * cnt / total if total else 0.0
    thin = cnt < THIN_CLASS_THRESHOLD
    rows.append({'Case': case, 'Count': cnt, 'Pct': pct,
                 'Thin': '⚠ thin' if thin else ''})

# Also report any unexpected case values
known = {'Nom', 'Acc', 'Ins', 'Dat', 'Abl', 'Gen', 'Loc', 'Voc'}
other = {k: v for k, v in case_counter.items() if k not in known}

print('══════════════════════════════════════════')
print('  CASE DISTRIBUTION  (CHECKPOINT — READ ME)')
print('══════════════════════════════════════════')
print(f'  {"Case":<6}  {"Count":>7}  {"Pct":>6}  Status')
print('  ──────────────────────────────────────')
for r in rows:
    print(f"  {r['Case']:<6}  {r['Count']:>7,}  {r['Pct']:>5.1f}%  {r['Thin']}")
print('  ──────────────────────────────────────')
print(f'  TOTAL   {total:>7,}')
if other:
    print(f'  Other case values in data: {dict(other)}')
print()

thin_cases  = [r['Case'] for r in rows if r['Thin'] and r['Count'] > 0]
empty_cases = [r['Case'] for r in rows if r['Count'] == 0]
robust_cases = [r['Case'] for r in rows if not r['Thin'] and r['Count'] > 0]

print('RECOMMENDATION')
print(f'  Robust (≥{THIN_CLASS_THRESHOLD}) : {robust_cases}')
if thin_cases:
    print(f'  Thin   (<{THIN_CLASS_THRESHOLD}) : {thin_cases}')
    print('  → Consider merging thin classes or dropping from probing.')
if empty_cases:
    print(f'  Absent (0)       : {empty_cases}')
    print('  → These classes have no data in this DCS eval subset.')
print()
print('⚑ Review the table above. When satisfied, proceed to Cell 5.')
print('══════════════════════════════════════════')

# Write to disk
counts_path = os.path.join(OUTPUT_DIR, 'case_counts.csv')
pd.DataFrame(rows).to_csv(counts_path, index=False)
print(f'Counts saved → {counts_path}')

---
## Cell 5 · Load Trained Model
Loads `best_checkpoint.pt` and reconstructs the exact `SanskritLM` architecture.
The model definition is copied verbatim from `sanskrit_lm_v1.ipynb` Cell 3 so
the weights load without any key mismatches.

In [ ]:
# ── Model definition (verbatim from sanskrit_lm_v1.ipynb Cell 3) ──────────────
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head   = n_head
        self.head_dim = n_embd // n_head
        self.dropout  = dropout
        self.c_attn   = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj   = nn.Linear(n_embd, n_embd, bias=False)
        self.resid_drop = nn.Dropout(dropout)
        self.use_flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.use_flash:
            self.register_buffer(
                'mask', torch.tril(torch.ones(block_size, block_size))
                              .unsqueeze(0).unsqueeze(0)
            )

    def forward(self, x):
        B, T, C = x.shape
        q, k, v  = self.c_attn(x).split(C, dim=2)
        reshape  = lambda t: t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k, v  = reshape(q), reshape(k), reshape(v)
        if self.use_flash:
            out = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=self.dropout if self.training else 0.0, is_causal=True,
            )
        else:
            scale = self.head_dim ** -0.5
            att   = (q @ k.transpose(-2, -1)) * scale
            att   = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
            att   = torch.softmax(att.float(), dim=-1).to(q.dtype)
            out   = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(out))


class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=False),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd, bias=False),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2  = nn.LayerNorm(n_embd)
        self.mlp  = MLP(n_embd, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class SanskritLM(nn.Module):
    def __init__(self, vocab_size, n_layer, n_head, n_embd, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.ModuleList([
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None, return_hidden_states=False):
        B, T = idx.shape
        assert T <= self.block_size
        pos = torch.arange(T, device=idx.device)
        x   = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        hidden_states = []
        for block in self.blocks:
            x = block(x)
            if return_hidden_states:
                hidden_states.append(x.detach().clone())
        x      = self.ln_f(x)
        logits = self.head(x)
        loss   = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        if return_hidden_states:
            return logits, loss, hidden_states
        return logits, loss


# ── Load checkpoint ────────────────────────────────────────────────────────────
print(f'Loading checkpoint: {MODEL_CKPT_PATH}')
ckpt = torch.load(MODEL_CKPT_PATH, map_location=device)
cfg  = ckpt['config']

# Vocab: prefer VOCAB_PATH (standalone JSON), fall back to vocab embedded in checkpoint
if os.path.exists(VOCAB_PATH):
    with open(VOCAB_PATH, encoding='utf-8') as f:
        vd = json.load(f)
    stoi = vd['stoi']
    itos = {int(k): v for k, v in vd['itos'].items()}
    print(f'Vocab loaded from {VOCAB_PATH}  ({len(stoi)} chars)')
else:
    stoi = ckpt.get('vocab_stoi', {})
    itos = {int(k): v for k, v in ckpt.get('vocab_itos', {}).items()}
    print(f'Vocab loaded from checkpoint  ({len(stoi)} chars)')

model = SanskritLM(
    vocab_size = cfg['vocab_size'],
    n_layer    = cfg['N_LAYER'],
    n_head     = cfg['N_HEAD'],
    n_embd     = cfg['N_EMBD'],
    block_size = cfg['BLOCK_SIZE'],
    dropout    = 0.0,
).to(device)
model.load_state_dict(ckpt['model'])
model.eval()

N_LAYERS   = cfg['N_LAYER']
N_EMBD     = cfg['N_EMBD']
BLOCK_SIZE = cfg['BLOCK_SIZE']

# Now that we know BLOCK_SIZE, tighten the guard
_BLOCK_SIZE_GUARD = BLOCK_SIZE

# Resolve LAYERS config: None → all layers
if LAYERS is None:
    LAYERS = list(range(N_LAYERS))

print(f'Model  : {N_LAYERS}L × {cfg["N_HEAD"]}H × {N_EMBD}D  block_size={BLOCK_SIZE}')
print(f'Layers : {LAYERS}')
print(f'Params : {sum(p.numel() for p in set(model.parameters()))/1e6:.2f}M unique')
print(f'Iter   : {ckpt["iter"]:,}  val_loss={ckpt.get("best_val_loss", float("nan")):.4f}')

---
## Cell 6 · Extract Contextual Activations
For each sentence:
1. Encode the full SLP1 sentence to character IDs (using the training vocab)
2. Run the model with `return_hidden_states=True` → 6 residual-stream tensors
3. For each case-bearing target token, extract at its encoded position range:
   - `final_char`: activation at the token's last character position
   - `mean_pool`: mean over the token's character positions
4. Accumulate results; skip tokens where any character is OOV

No gradients; model frozen in eval mode throughout.

In [ ]:
# Pre-filter sentences: re-apply the exact block_size guard now that we know it,
# and remove sentences whose token spans had OOV characters.
sentences_to_process = [
    e for e in all_parsed_sentences
    if len(e['slp1_text']) <= BLOCK_SIZE
]
n_dropped_long = len(all_parsed_sentences) - len(sentences_to_process)
print(f'Sentences to process: {len(sentences_to_process):,}')
if n_dropped_long:
    print(f'  (Dropped {n_dropped_long} sentences exceeding block_size={BLOCK_SIZE})')

# ── Accumulators ──────────────────────────────────────────────────────────────
# Lists indexed by target token order. Shape per entry: (N_LAYERS, N_EMBD)
fc_accum   = []   # final_char activations
mp_accum   = []   # mean_pool activations
meta_rows  = []   # metadata, one dict per valid target token

# Skip counters
n_skipped_oov        = 0
n_skipped_span_empty = 0
n_extracted          = 0

# ── Extraction loop ────────────────────────────────────────────────────────────
t_start = time.time()
log_interval = max(1, len(sentences_to_process) // 20)

with torch.no_grad():
    for sent_idx, entry in enumerate(sentences_to_process):

        if not entry['targets']:
            continue

        slp1_text = entry['slp1_text']
        ids, char_to_pos = encode_aligned(slp1_text, stoi)

        if not ids:
            n_skipped_oov += len(entry['targets'])
            continue

        # Check for OOV characters in any target's span.
        # We log and skip individual tokens, not whole sentences.
        tok_positions = []   # (tok_start, tok_end_inclusive) or None
        for t in entry['targets']:
            pos = span_to_positions(t['char_start'], t['char_end'], char_to_pos)
            if pos is None:
                n_skipped_oov += 1
                tok_positions.append(None)
            else:
                tok_positions.append(pos)

        # If ALL targets are OOV, skip the forward pass
        if all(p is None for p in tok_positions):
            continue

        # Forward pass: (1, T) → list of (1, T, D) hidden states
        ids_tensor = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
        _, _, hidden_states = model(ids_tensor, return_hidden_states=True)
        # hidden_states[l]: (1, T, D) — still on GPU

        # Stack all layers into a single (N_LAYERS, T, D) float32 tensor on CPU
        # to avoid repeated GPU→CPU transfers inside the inner loop
        T_seq = ids_tensor.shape[1]
        all_layers = torch.stack(
            [hidden_states[l][0] for l in LAYERS], dim=0
        ).cpu().float()  # (N_LAYERS, T, D)

        # Extract per-target activations
        for t, pos in zip(entry['targets'], tok_positions):
            if pos is None:
                continue
            tok_s, tok_e = pos  # inclusive

            if tok_e >= T_seq:
                n_skipped_span_empty += 1
                continue

            # final_char: activation at last char of the token — shape (N_LAYERS, D)
            fc_vec = all_layers[:, tok_e, :]                        # (N_LAYERS, D)

            # mean_pool: mean over token's char span — shape (N_LAYERS, D)
            if tok_s == tok_e:
                mp_vec = fc_vec
            else:
                mp_vec = all_layers[:, tok_s:tok_e+1, :].mean(dim=1)  # (N_LAYERS, D)

            fc_accum.append(fc_vec.numpy())    # (N_LAYERS, D)
            mp_accum.append(mp_vec.numpy())
            meta_rows.append({
                'token_row_id': n_extracted,
                'sent_id'     : t['sent_id'],
                'source_file' : os.path.basename(t['source_file']),
                'token_id'    : t['token_id'],
                'token_slp1'  : t['form_slp1'],
                'lemma'       : t['lemma'],
                'upos'        : t['upos'],
                'case_label'  : t['case_label'],
                'full_feats'  : t['full_feats'],
                'char_start'  : t['char_start'],
                'char_end'    : t['char_end'],
                'tok_pos_start': tok_s,
                'tok_pos_end'  : tok_e,
            })
            n_extracted += 1

        if (sent_idx + 1) % log_interval == 0:
            elapsed = time.time() - t_start
            rate    = (sent_idx + 1) / elapsed
            eta     = (len(sentences_to_process) - sent_idx - 1) / rate
            print(f'  [{sent_idx+1:>6}/{len(sentences_to_process)}]  '
                  f'extracted={n_extracted:,}  '
                  f'elapsed={elapsed:.0f}s  ETA={eta:.0f}s')

wall = time.time() - t_start
print(f'\nExtraction complete in {wall:.1f}s  ({wall/60:.1f} min)')
print(f'  Extracted  : {n_extracted:,} target tokens')
print(f'  Skipped    : {n_skipped_oov:,} (OOV char in span)')
print(f'  Skipped    : {n_skipped_span_empty:,} (span out of sequence range)')

---
## Cell 7 · Assemble and Save Dataset
Writes four files to `OUTPUT_DIR`:
- `probing_metadata.csv` — one row per target token, all metadata columns
- `probing_activations.npz` — `final_char` and `mean_pool` arrays, shape `(N_tokens, N_layers, D)`
- `case_counts.csv` — per-class counts (already saved in Cell 4)
- `manifest.json` — provenance record

**Loading the dataset in the probing step:**
```python
import numpy as np, pandas as pd
meta = pd.read_csv('probing_metadata.csv')
acts = np.load('probing_activations.npz')  # keys: 'final_char', 'mean_pool'
# acts['final_char'][i, layer, :]  →  512-dim vector for token i at layer 'layer'
```

In [ ]:
import datetime

if n_extracted == 0:
    raise RuntimeError('No tokens were extracted. Check Cell 3 and Cell 6 for errors.')

# ── Stack activation arrays ────────────────────────────────────────────────────
# Each entry in fc_accum/mp_accum is (N_LAYERS, D) float32 numpy array
print('Stacking activation arrays ...')
fc_array = np.stack(fc_accum, axis=0).astype(np.float32)  # (N_tokens, N_layers, D)
mp_array = np.stack(mp_accum, axis=0).astype(np.float32)
print(f'  final_char : {fc_array.shape}  {fc_array.nbytes/1e6:.1f} MB')
print(f'  mean_pool  : {mp_array.shape}  {mp_array.nbytes/1e6:.1f} MB')

# ── Metadata DataFrame ─────────────────────────────────────────────────────────
meta_df = pd.DataFrame(meta_rows)

# ── Per-class counts (from extracted set — may differ from Cell 4 if some skipped) ─
final_case_counts = meta_df['case_label'].value_counts().to_dict()

# ── Save ───────────────────────────────────────────────────────────────────────
meta_path = os.path.join(OUTPUT_DIR, 'probing_metadata.csv')
acts_path = os.path.join(OUTPUT_DIR, 'probing_activations.npz')
mfst_path = os.path.join(OUTPUT_DIR, 'manifest.json')

meta_df.to_csv(meta_path, index=False)
np.savez_compressed(acts_path, final_char=fc_array, mean_pool=mp_array)

manifest = {
    'created_at'            : datetime.datetime.utcnow().isoformat() + 'Z',
    'model_checkpoint'      : MODEL_CKPT_PATH,
    'model_iter'            : int(ckpt['iter']),
    'model_val_loss'        : float(ckpt.get('best_val_loss', float('nan'))),
    'model_arch'            : {
        'n_layer'     : N_LAYERS,
        'n_head'      : cfg['N_HEAD'],
        'n_embd'      : N_EMBD,
        'block_size'  : BLOCK_SIZE,
    },
    'dcs_source'            : DCS_CONLLU_PATH,
    'n_sentences_parsed'    : n_sentences_total,
    'n_sentences_processed' : len(sentences_to_process),
    'n_target_tokens'       : n_extracted,
    'n_layers_extracted'    : len(LAYERS),
    'layers'                : LAYERS,
    'position_variants'     : POSITION_VARIANTS,
    'activation_shape'      : list(fc_array.shape),  # (N_tokens, N_layers, D)
    'activation_dtype'      : 'float32',
    'per_class_counts'      : final_case_counts,
    'thin_class_threshold'  : THIN_CLASS_THRESHOLD,
    'thin_classes'          : [c for c, n in final_case_counts.items()
                               if n < THIN_CLASS_THRESHOLD],
    'skipped_oov'           : n_skipped_oov,
    'skipped_mwt_sub'       : skip_log.get('mwt_sub (case inside MWT)', 0),
    'skipped_long_sentence' : n_sentences_skipped_too_long,
    'vocab_size'            : len(stoi),
    'outputs': {
        'metadata'    : meta_path,
        'activations' : acts_path,
        'case_counts' : os.path.join(OUTPUT_DIR, 'case_counts.csv'),
        'manifest'    : mfst_path,
    },
}
with open(mfst_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Saved:')
print(f'  {meta_path}   ({len(meta_df):,} rows)')
print(f'  {acts_path}   ({os.path.getsize(acts_path)/1e6:.1f} MB compressed)')
print(f'  {mfst_path}')

---
## Cell 8 · Status Report

In [ ]:
print('═' * 68)
print('PROBING DATASET STATUS')
print('═' * 68)
print(f'  Model checkpoint : {MODEL_CKPT_PATH}')
print(f'  Trained iters    : {ckpt["iter"]:,}')
print(f'  Best val loss    : {ckpt.get("best_val_loss", float("nan")):.4f}')
print()
print(f'  DCS source       : {DCS_CONLLU_PATH}')
print(f'  Sentences parsed : {n_sentences_total:,}')
print(f'  Sentences used   : {len(sentences_to_process):,}')
print(f'  Dropped (length) : {n_dropped_long:,} (>{BLOCK_SIZE} chars)')
print()
print(f'  Target tokens    : {n_extracted:,} extracted')
print(f'  Skipped OOV      : {n_skipped_oov:,}')
print(f'  Skipped MWT sub  : {skip_log.get("mwt_sub (case inside MWT)", 0):,}')
print()
print(f'  Activation shape : {list(fc_array.shape)} = (tokens, layers, embd_dim)')
print(f'  Layers           : {LAYERS}')
print(f'  Variants         : {POSITION_VARIANTS}')
print()
print('  PER-CLASS COUNTS (extracted set):')
for case in ['Nom', 'Acc', 'Ins', 'Dat', 'Abl', 'Gen', 'Loc', 'Voc']:
    cnt = final_case_counts.get(case, 0)
    bar = '█' * min(cnt // 200, 30)
    thin_tag = '  ⚠ thin' if cnt < THIN_CLASS_THRESHOLD else ''
    print(f'    {case:<4} {cnt:>7,}  {bar}{thin_tag}')
print()
print('  SPAN SANITY CHECK (5 examples printed in Cell 3)')
print('  (Re-run Cell 3 display block to review more examples)')
print()
print('  OUTPUT FILES:')
for k, v in manifest['outputs'].items():
    print(f'    {k:<14} {v}')
print()
print('  NEXT STEPS:')
print('    1. Train linear probes (logistic regression per layer, per Case)')
print('    2. Plot per-layer probe accuracy to see which layer encodes Case')
print('    3. PCA/UMAP on activations coloured by case_label for visual evidence')
print('═' * 68)